# 16 — Flip / Normalization Order Audit

Downloads the two W&B artifacts produced by `diffusionsr/analysis/audit_flip_norm.py`
(one for single-field / temperature-only, one for multifield / temperature+sdfliqlabel)
and renders all figures and summary metrics.

**This notebook needs no local data or model checkpoints.**  
Run `audit_flip_norm.py` on TRACE first, then re-run this notebook locally.

See `AUDIT_FLIP_NORM.md` for the full code-analysis report.

In [ ]:
import os, json, glob
import wandb
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
from IPython.display import display

# ── CONFIG ─────────────────────────────────────────────────────────────────
WANDB_ENTITY  = os.getenv('WANDB_ENTITY', 'your-entity')
WANDB_PROJECT = 'Flow3D_SuperResolution'
ARTIFACT_TAGS = ['single_field', 'multifield']   # matches --tag in audit_flip_norm.py
LOCAL_DIR     = '/tmp/audit_flip_norm'           # where to download artifact files
# ───────────────────────────────────────────────────────────────────────────

os.makedirs(LOCAL_DIR, exist_ok=True)

In [ ]:
# Download both artifacts from W&B
api = wandb.Api()
artifact_dirs = {}

for tag in ARTIFACT_TAGS:
    art_name = f'{WANDB_ENTITY}/{WANDB_PROJECT}/audit_flip_norm_{tag}:latest'
    try:
        art = api.artifact(art_name)
        dl_dir = art.download(root=os.path.join(LOCAL_DIR, tag))
        artifact_dirs[tag] = dl_dir
        print(f'[{tag}] downloaded → {dl_dir}')
    except Exception as e:
        print(f'[{tag}] MISSING artifact {art_name!r}: {e}')

In [ ]:
# Load both JSON summaries
summaries = {}
for tag, dl_dir in artifact_dirs.items():
    json_path = os.path.join(dl_dir, f'{tag}_summary.json')
    if os.path.exists(json_path):
        with open(json_path) as f:
            summaries[tag] = json.load(f)
        print(f'[{tag}] summary loaded')
    else:
        print(f'[{tag}] WARNING: {json_path} not found')

## Effect Size: NtF vs FtN (Kelvin)

In [ ]:
# Build a comparison table: single-field vs multifield
rows = []
for tag, s in summaries.items():
    es = s.get('effect_size', {})
    rows.append({
        'Tag':              tag,
        'Fields':           ', '.join(s.get('field_names', [])),
        'N samples':        es.get('n_samples'),
        'Mean err (norm)':  f"{es.get('mean_err_norm', float('nan')):.4f}",
        'Mean err (K)':     f"{es.get('mean_err_K', float('nan')):.1f}",
        'MP mean err (K)':  f"{es.get('mean_err_mp_K', float('nan')):.1f}",
        'BG mean err (K)':  f"{es.get('mean_err_bg_K', float('nan')):.1f}",
        'p95 err (K)':      f"{es.get('p95_err_K', float('nan')):.1f}",
    })

df_effect = pd.DataFrame(rows)
display(df_effect)

## Statistics Asymmetry

In [ ]:
# Asymmetry table for all fields in each tag
asym_rows = []
for tag, s in summaries.items():
    for stat_key, fields_dict in s.get('asymmetry', {}).items():
        for fname, vals in fields_dict.items():
            asym_rows.append({
                'Tag':              tag,
                'Statistic':        stat_key,
                'Field':            fname,
                'Max |diff| (K)':   f"{vals.get('max_abs_diff_K', float('nan')):.1f}",
                'Mean |diff| (K)':  f"{vals.get('mean_abs_diff_K', float('nan')):.1f}",
                'Max |diff|/std':   f"{vals.get('max_rel_diff', float('nan')):.3f}",
                'Mean |diff|/std':  f"{vals.get('mean_rel_diff', float('nan')):.3f}",
            })

df_asym = pd.DataFrame(asym_rows)
display(df_asym)

## Figures — Single Field

In [ ]:
def show_pngs(tag, pattern_fragment, title_prefix=''):
    dl_dir = artifact_dirs.get(tag, '')
    paths  = sorted(glob.glob(os.path.join(dl_dir, f'*{pattern_fragment}*.png')))
    if not paths:
        print(f'No figures matching *{pattern_fragment}* in {dl_dir}')
        return
    for p in paths:
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.imshow(mpimg.imread(p))
        ax.axis('off')
        ax.set_title(f'{title_prefix}{os.path.basename(p)}', fontsize=9)
        plt.tight_layout()
        plt.show()

show_pngs('single_field', 'asymmetry', '[single] ')

In [ ]:
show_pngs('single_field', 'effect_size',      '[single] ')
show_pngs('single_field', 'example_error_map','[single] ')

## Figures — Multifield

In [ ]:
show_pngs('multifield', 'asymmetry', '[multi] ')

In [ ]:
show_pngs('multifield', 'effect_size',       '[multi] ')
show_pngs('multifield', 'example_error_map', '[multi] ')

## Overlay: Single-Field vs Multifield Effect Size

In [ ]:
import numpy as np

cats = ['Overall', 'Melt pool', 'Background']
keys = ['mean_err_K', 'mean_err_mp_K', 'mean_err_bg_K']

x  = np.arange(len(cats))
w  = 0.35
fig, ax = plt.subplots(figsize=(7, 4), dpi=120)

colors = {'single_field': '#4c72b0', 'multifield': '#dd8452'}
for i, tag in enumerate(['single_field', 'multifield']):
    es = summaries.get(tag, {}).get('effect_size', {})
    values = [es.get(k, float('nan')) for k in keys]
    bars = ax.bar(x + i*w, values, w, label=tag.replace('_', ' '),
                  color=colors[tag], edgecolor='w', alpha=0.9)
    for bar, val in zip(bars, values):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f'{val:.1f}', ha='center', fontsize=8)

ax.set_xticks(x + w/2); ax.set_xticklabels(cats)
ax.set_ylabel('Mean |NtF − FtN| error (K)')
ax.set_title('Ordering error by region — single-field vs multifield')
ax.legend()
plt.tight_layout()
plt.show()

## Raw JSON Summaries

In [ ]:
for tag, s in summaries.items():
    print(f'\n=== {tag} ===')
    print(json.dumps(s, indent=2))